## Analytic Power Spectrum Calculation

Set up the analytic framework using lognormal fields and Hankel transforms to compute dark photon auto-power spectra.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
import pickle
sys.path.append("../")

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import numpy as np

# Special functions for analytic calculations
from scipy.special import spherical_jn, hyp2f1
from scipy.interpolate import interp1d

# Dark photon perturbation theory code
grf_path = "/home/bakerem/dark-photons-perturbations"
sys.path.append(grf_path)

from grf.grf import PerturbedProbability, FIRAS
from grf.pk_interp import PowerSpectrumGridInterpolator
from grf.units import *

from IPython.display import set_matplotlib_formats
set_matplotlib_formats('retina')

%matplotlib inline

# Custom plotting parameters
from plot_params import params
pylab.rcParams.update(params)

cols_default = plt.rcParams['axes.prop_cycle'].by_key()['color']

In [ ]:
# Load nonlinear matter power spectrum with baryon physics
log_pspec = PowerSpectrumGridInterpolator("franken_lower")

# Initialize perturbation theory classes
prob = PerturbedProbability(log_pspec)
firas = FIRAS(log_pspec)

# Matter power spectrum integration bounds
k_min = 1e-6  # Mpc^-1
k_max = 1e6   # Mpc^-1

# Frequency for conversion calculation (115 MHz ~ z=11.3 for 21cm)
nu_in_GHz = 0.410
omega_0 = 2 * np.pi * nu_in_GHz * 1e9 * Hz  # Convert to natural units

def compute_Cls(m_Ap):
    """
    Compute angular power spectrum for given dark photon mass using analytic methods.
    
    Uses lognormal approximation for density fluctuations and Hankel transforms
    for efficient computation of correlation functions.
    """
    
    # Dark photon-photon coupling (FIRAS bound)
    eps = 1e-6
    
    # Maximum redshift matching halo simulations
    z_star = 4.
    
    # Redshift array for integration
    z_ary = np.geomspace(0.005, z_star, 1000)
    
    # Plasma mass squared evolution
    m_A_sq_ary = firas.m_A_sq(z_ary, omega_0)
    
    # Variance of density fluctuations (lognormal parameterization)
    sigma_1_sq_ary = firas._dP_dz(z_ary, m_Ap, k_min, k_max, omega_0, pdf='lognormal')[1][0]
    sigma_sq_ary = np.exp(sigma_1_sq_ary) - 1.
    
    # Mass ratio parameter g = m_A² / ⟨m_γ²⟩ - 1
    g_ary = m_Ap**2 / m_A_sq_ary - 1
    
    # Total conversion probability (for normalization)
    total_prob = np.trapz(firas._dP_dz(z_ary, m_Ap, k_min, k_max, omega_0, pdf='lognormal')[0][0], z_ary)
    
    # Fast Fourier-Bessel transform setup
    import pyfftlog 
    
    def fftj0(f, logrmin, logrmax, n_pts=4096, q=0):
        """
        Hankel transform (Fourier-Bessel) for spherically symmetric functions.
        
        Computes ∫ d³r a(r) j₀(kr) efficiently using FFTLog algorithm.
        """
        # Configuration for optimal performance
        kr = 1  # Central k*r value
        kropt = 1  # Low-ringing optimization
        tdir = 1  # Forward transform
        
        # Set up logarithmic grid
        logrc = (logrmin + logrmax)/2
        nc = (n_pts + 1)/2.0
        dlogr = (logrmax - logrmin)/n_pts
        dlnr = dlogr*np.log(10.0)
        
        # Initialize FFTLog
        kr, xsave = pyfftlog.fhti(n_pts, 0.5, dlnr, q, kr, kropt)
        logkc = np.log10(kr) - logrc
        
        # Create coordinate arrays
        r_ary = 10**(logrc + (np.arange(1, n_pts+1) - nc)*dlogr)
        k_ary = 10**(logkc + (np.arange(1, n_pts+1) - nc)*dlogr)
        
        # Apply function and transform
        ar_ary = np.moveaxis(f(r_ary), 0, -1) * (r_ary)**(1.5 - q)
        ak_ary = np.zeros(ar_ary.shape)
        
        # Handle multidimensional arrays
        if len(ak_ary.shape) > 1:
            indices_ary = np.moveaxis(np.indices(ak_ary[...,0].shape), 0, -1)
            for ind in indices_ary.reshape(-1, indices_ary.shape[-1]):
                if ind.shape == ():
                    ak_ary[ind] = (2*np.pi)**1.5 * k_ary**(-1.5-q) * pyfftlog.fht(ar_ary[ind].copy(), xsave, tdir)
                else:
                    ak_ary[tuple(ind)] = (2*np.pi)**1.5 * k_ary**(-1.5-q) * pyfftlog.fht(ar_ary[tuple(ind)].copy(), xsave, tdir)
        else:
            ak_ary = (2*np.pi)**1.5 * k_ary**(-1.5-q) * pyfftlog.fht(ar_ary.copy(), xsave, tdir)
        
        return (k_ary, ak_ary)
    
    # Matter power spectrum function for Hankel transform
    def p_spec_out(k_ary): 
        return np.transpose(10**log_pspec(z_ary, k_ary))
    
    # Compute correlation function via Hankel transform
    r_corr_func_fft_ary, corr_func_fft_ary = fftj0(p_spec_out, np.log10(k_min), np.log10(k_max))
    corr_func_fft_ary /= (2 * np.pi)**3 
    
    # Create interpolation function
    corr_func_fft = interp1d(r_corr_func_fft_ary, corr_func_fft_ary)
    
    # Cosmological parameters
    Omega_Lambda = firas.cosmo.Ode0
    Omega_m = firas.cosmo.Odm0 + firas.cosmo.Ob0
    H0_in_per_Mpc = firas.cosmo.H(0).value * Kmps
    h = firas.cosmo.h
    
    def comoving_dist(z):
        """Comoving distance calculation with hypergeometric functions."""
        def indef_integral(y):
            return ((1. + y) / (H0_in_per_Mpc * Omega_Lambda) 
                   * np.sqrt(Omega_Lambda + Omega_m * (1 + y)**3)
                   * hyp2f1(5/6, 1., 4/3, -Omega_m*(1+y)**3 / Omega_Lambda))
        
        return (indef_integral(z) - indef_integral(0)) * h
    
    # Comoving distances
    chi_ary = comoving_dist(z_ary)
    
    # Set up correlation function calculation
    r_ary = np.logspace(-5, 5, num=700)
    r_ary_for_xi = np.outer(r_ary, np.ones_like(z_ary))
    
    # Correlation function from Hankel transform
    xi_ary = np.transpose(corr_func_fft(r_ary))
    
    # Lognormal field parameters
    L_ary = np.log1p(g_ary) + sigma_1_sq_ary / 2
    X_ary = np.log1p(xi_ary)
    
    # Conversion probability fluctuations (lognormal model)
    expr_to_fft_full_ary = (
        1. / np.sqrt(1 - X_ary**2 / sigma_1_sq_ary**2)
        * np.exp(-L_ary**2 / (sigma_1_sq_ary + X_ary)) - np.exp(-L_ary**2 / sigma_1_sq_ary)
    )
    
    # Keep only positive contributions
    expr_to_fft_ary = np.zeros_like(expr_to_fft_full_ary)
    expr_to_fft_ary[expr_to_fft_full_ary > 0] = expr_to_fft_full_ary[expr_to_fft_full_ary > 0]
    
    # Transform to angular space
    func_to_fft = interp1d(r_ary, expr_to_fft_ary, axis=0)
    fft_res = fftj0(func_to_fft, np.log10(r_ary[0]), np.log10(r_ary[-1]))
    
    l_over_chi_ary = fft_res[0]
    integrand = fft_res[1]
    
    # Interpolation for angular power spectrum
    r_integral = interp1d(l_over_chi_ary, integrand, bounds_error=False, fill_value=(np.nan, integrand[:,-1]))
    
    # Mean conversion probability
    P_mean = eps**2 * np.trapz(firas._dP_dz(z_ary, m_Ap, k_min, k_max, omega_0, pdf='lognormal')[0][0], z_ary)
    
    def C_l(l): 
        """Angular power spectrum at multipole l."""
        # Hubble parameter in Mpc^-1
        hubble_ary = firas.cosmo.H(z_ary).value * Kmps / h
        
        # Radial integral evaluation
        r_int = np.diag(r_integral(l/comoving_dist(z_ary)))
        
        # Physical prefactor (units: Mpc^2)
        prefac = (np.pi * m_Ap**4 * eps**2 / omega_0)**2 * Mpc**2 / h**2
        
        # Full integrand for C_l
        integrand = (1. / comoving_dist(z_ary)**2 / hubble_ary / (1. + z_ary)**4 / 
                    (2 * np.pi * sigma_1_sq_ary * m_A_sq_ary**2 * (1. + g_ary)**2) * r_int)
        
        return np.trapezoid(prefac*integrand, z_ary)
    
    # Compute power spectrum for range of multipoles
    l_ary = np.geomspace(10, 4000, 5000, dtype=int)
    C_l_ary = np.array([C_l(l) for l in l_ary])
    
    return l_ary, C_l_ary

## Compute and Visualize Power Spectra

In [ ]:
# Compute power spectra for two representative dark photon masses
Cls = []
mA_list = [3e-14, 8e-14]  # eV

for mA in mA_list:
    l_ary, Cl = compute_Cls(mA)
    Cls.append(Cl)

In [ ]:
# Plot analytic angular power spectra
labels = [r"$m_{A'}=3\times 10^{-14}$ eV", r"$m_{A'}=8\times 10^{-14}$ eV"]
Tgamma0 = 2.73e3  # CMB monopole temperature in mK

for i in range(len(Cls)):
    # Convert to observed brightness temperature units
    plt.loglog(l_ary, Tgamma0**2 * Cls[i], label=labels[i], color=cols_default[i])

plt.legend()
plt.xlim(100, 4000)
plt.xlabel(r"$\ell$")
plt.ylabel(r"$C_\ell^{\rm aa}$ [mK$^2$]")
plt.savefig("plots/analytic_Cls.pdf", bbox_inches='tight')